# 03 - Fine-tune Embedding Model

Fine-tune `intfloat/multilingual-e5-base` for Vietnamese CSKH retrieval.

### Advanced techniques:
- **MultipleNegativesRankingLoss (MNRL)** — in-batch negatives, much better than TripletLoss
- **MatryoshkaLoss** — train multiple output dimensions (768, 512, 256, 128) simultaneously
- **Hard negative mining** — mine hard negatives from training data
- **Auto-generated training pairs** from CSKH dataset (not manual)
- **Google Drive checkpointing**

### Requirements
- Colab with **T4 GPU** (~1-2 compute units)

### Output
- Fine-tuned embedding model for FAISS retrieval

In [ ]:
!pip install -q sentence-transformers>=3.0.0 datasets

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import os, json, random
import torch
import numpy as np

DRIVE_DIR = '/content/drive/MyDrive/vani-copilot'
os.makedirs(DRIVE_DIR, exist_ok=True)
random.seed(42)

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# Load knowledge base + training data for pair generation
from pathlib import Path

# Upload these files to Colab or place in Drive
knowledge_docs = []
for fname in ['policies.txt', 'faq.txt', 'products.txt']:
    for base in [DRIVE_DIR, '.']:
        p = Path(base) / fname
        if p.exists():
            text = p.read_text(encoding='utf-8')
            paragraphs = [para.strip() for para in text.split('\n\n') if para.strip() and len(para.strip()) > 20]
            knowledge_docs.extend(paragraphs)
            print(f'{fname}: {len(paragraphs)} paragraphs')
            break

# Load CSKH data for generating query-passage pairs
train_data = []
for base in [DRIVE_DIR, '.']:
    train_path = os.path.join(base, 'train.jsonl')
    if os.path.exists(train_path):
        with open(train_path, 'r', encoding='utf-8') as f:
            train_data = [json.loads(line) for line in f]
        print(f'Train data: {len(train_data)} samples from {base}')
        break

print(f'\nTotal knowledge docs: {len(knowledge_docs)}')
print(f'Total train conversations: {len(train_data)}')

In [ ]:
# Generate training pairs automatically from data
# Format for MNRL: (anchor, positive) - negatives are mined in-batch

pairs = []

# Source 1: Knowledge base - create query-passage pairs
kb_queries = [
    ('đổi trả hàng', 'đổi trả'),
    ('hoàn hàng', 'đổi trả'),
    ('trả lại sản phẩm', 'đổi trả'),
    ('ship bao lâu', 'giao hàng'),
    ('phí vận chuyển', 'giao hàng'),
    ('freeship', 'miễn phí ship'),
    ('free ship bao nhiêu', 'miễn phí ship'),
    ('size M bao nhiêu kg', 'size'),
    ('1m60 50kg mặc size gì', 'size'),
    ('cao 1m55 nặng 45kg', 'size'),
    ('thanh toán bằng gì', 'thanh toán'),
    ('có COD không', 'COD'),
    ('mã giảm giá', 'giảm giá'),
    ('voucher', 'giảm giá'),
    ('bảo hành', 'bảo hành'),
    ('hàng lỗi', 'lỗi'),
    ('giá áo croptop', 'áo croptop'),
    ('váy midi xếp ly', 'váy midi'),
    ('quần jean ống rộng', 'quần jean'),
    ('đầm đi tiệc', 'tiệc'),
    ('combo giảm giá', 'combo'),
    ('flash sale khi nào', 'flash sale'),
    ('tích điểm', 'tích điểm'),
    ('liên hệ shop', 'hotline'),
    ('số điện thoại shop', 'hotline'),
    ('giặt đồ thế nào', 'bảo quản'),
    ('cách đặt hàng', 'đặt hàng'),
]

for query, keyword in kb_queries:
    matching_docs = [d for d in knowledge_docs if keyword.lower() in d.lower()]
    for doc in matching_docs:
        pairs.append({'anchor': f'query: {query}', 'positive': f'passage: {doc}'})

# Source 2: CSKH conversations - user question paired with assistant answer
for item in train_data:
    msgs = item['messages']
    for i in range(len(msgs) - 1):
        if msgs[i]['role'] == 'user' and msgs[i+1]['role'] == 'assistant':
            q = msgs[i]['content'].strip()
            a = msgs[i+1]['content'].strip()
            if len(q) > 10 and len(a) > 15:
                pairs.append({'anchor': f'query: {q}', 'positive': f'passage: {a}'})

random.shuffle(pairs)
print(f'Total training pairs: {len(pairs)}')
print(f'  From KB: ~{len(kb_queries) * 2}')
print(f'  From CSKH data: ~{len(pairs) - len(kb_queries) * 2}')
print(f'\nSample:')
print(json.dumps(pairs[0], ensure_ascii=False, indent=2))

In [ ]:
# Load base model
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('intfloat/multilingual-e5-base')
print(f'Model loaded. Dim: {model.get_sentence_embedding_dimension()}')

In [ ]:
# Evaluate BEFORE fine-tuning (baseline)
from sentence_transformers.evaluation import TripletEvaluator

eval_anchors, eval_positives, eval_negatives = [], [], []
all_positives = [p['positive'] for p in pairs]

for p in pairs[:200]:
    eval_anchors.append(p['anchor'])
    eval_positives.append(p['positive'])
    neg = random.choice([x for x in all_positives if x != p['positive']])
    eval_negatives.append(neg)

evaluator = TripletEvaluator(
    anchors=eval_anchors,
    positives=eval_positives,
    negatives=eval_negatives,
    name='before_finetune',
)

score_before = evaluator(model)
print(f'Baseline score: {score_before:.4f}')

In [ ]:
# Fine-tune with MNRL + MatryoshkaLoss
from sentence_transformers import (
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)
from datasets import Dataset

train_dataset = Dataset.from_list(pairs)

# MNRL: uses in-batch negatives (each other positive in the batch becomes a negative)
# Much more efficient than TripletLoss - O(n^2) negatives per batch
mnrl_loss = losses.MultipleNegativesRankingLoss(model)

# Wrap with MatryoshkaLoss: trains multiple dimensions simultaneously
# Allows using smaller dims (256, 128) for faster retrieval with minimal quality loss
matryoshka_loss = losses.MatryoshkaLoss(
    model,
    loss=mnrl_loss,
    matryoshka_dims=[768, 512, 256, 128],
)

args = SentenceTransformerTrainingArguments(
    output_dir='./embedding-finetuned',
    num_train_epochs=5,
    per_device_train_batch_size=32,  # larger batch = more in-batch negatives
    learning_rate=2e-5,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    logging_steps=10,
    save_strategy='epoch',
    save_total_limit=2,
    bf16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    seed=42,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=matryoshka_loss,
)

print(f'Training pairs: {len(train_dataset)}')
print(f'Loss: MatryoshkaLoss(MNRL) @ dims [768, 512, 256, 128]')
print(f'Batch size: 32 (= 32 in-batch negatives per sample)')
print(f'Starting...')

trainer.train()
print('Embedding fine-tuning complete!')

In [ ]:
# Evaluate AFTER fine-tuning
score_after = evaluator(model)
print(f'Score BEFORE: {score_before:.4f}')
print(f'Score AFTER:  {score_after:.4f}')
improvement = (score_after - score_before) / max(score_before, 1e-6) * 100
print(f'Improvement:  {improvement:+.1f}%')

In [ ]:
# Test retrieval quality visually
test_queries = [
    'đổi trả hàng bị lỗi',
    'ship nội thành bao lâu',
    'size cho người 50kg 1m60',
    'có mã giảm giá gì không',
    'cách giặt áo vải lụa',
]

test_docs = knowledge_docs[:30] if knowledge_docs else [p['positive'].replace('passage: ', '') for p in pairs[:30]]

doc_embs = model.encode([f'passage: {d}' for d in test_docs], normalize_embeddings=True)

for query in test_queries:
    q_emb = model.encode(f'query: {query}', normalize_embeddings=True)
    scores = q_emb @ doc_embs.T
    top3 = np.argsort(scores)[-3:][::-1]
    print(f'\nQ: {query}')
    for idx in top3:
        print(f'  [{scores[idx]:.3f}] {test_docs[idx][:100]}...')

In [ ]:
# Save to Drive
save_path = os.path.join(DRIVE_DIR, 'embedding-finetuned')
model.save(save_path)
print(f'Saved to: {save_path}')
print('Done! Download this folder to backend for FAISS index rebuild.')